# Data Modeling with Apache Cassandra

**Udacity Data Engineering Nanodegree — Project 1**

Building a NoSQL analytics database for **Sparkify** — a music streaming startup — using Apache Cassandra.  
The pipeline reads date-partitioned CSV event files, denormalizes them into a single flat file, and loads the data into three query-optimized Cassandra tables.

---
## Part I — ETL Pipeline for Pre-Processing Event Files

Raw event logs live in `event_data/` as date-partitioned CSV files. This section:
1. Walks every CSV file and collects only **NextSong** events.
2. Writes a consolidated, denormalized CSV (`event_datafile_new.csv`) containing the 11 columns required for Cassandra ingestion.

The resulting CSV has the following schema:

| # | Column | Type |
|---|--------|------|
| 0 | artist | str |
| 1 | firstName | str |
| 2 | gender | str |
| 3 | itemInSession | int |
| 4 | lastName | str |
| 5 | length | float |
| 6 | level | str |
| 7 | location | str |
| 8 | sessionId | int |
| 9 | song | str |
| 10 | userId | int |

In [ ]:
import pandas as pd
import cassandra
from cassandra.cluster import Cluster
import os
import glob
import csv

In [ ]:
filepath = os.getcwd() + '/event_data'

file_path_list = glob.glob(os.path.join(filepath, '*.csv'))

print(f"Found {len(file_path_list)} event file(s):")
for f in sorted(file_path_list):
    print(f"  {os.path.basename(f)}")

In [ ]:
full_data_rows_list = []

for f in file_path_list:
    with open(f, 'r', encoding='utf8', newline='') as csvfile:
        csvreader = csv.reader(csvfile)
        next(csvreader)
        for line in csvreader:
            full_data_rows_list.append(line)

print(f"Total raw event rows collected: {len(full_data_rows_list)}")

# Raw CSV column indices used below:
#  0=artist  1=auth  2=firstName  3=gender  4=itemInSession  5=lastName
#  6=length  7=level  8=location  9=method  10=page  11=registration
#  12=sessionId  13=song  14=status  15=ts  16=userId

csv.register_dialect('myDialect', quoting=csv.QUOTE_ALL, skipinitialspace=True)

with open('event_datafile_new.csv', 'w', encoding='utf8', newline='') as f:
    writer = csv.writer(f, dialect='myDialect')
    writer.writerow([
        'artist', 'firstName', 'gender', 'itemInSession',
        'lastName', 'length', 'level', 'location', 'sessionId', 'song', 'userId'
    ])
    for row in full_data_rows_list:
        if row[10] == 'NextSong':
            writer.writerow((
                row[0], row[2], row[3], row[4], row[5],
                row[6], row[7], row[8], row[12], row[13], row[16]
            ))

print("event_datafile_new.csv created.")

In [ ]:
with open('event_datafile_new.csv', 'r', encoding='utf8') as f:
    row_count = sum(1 for _ in f) - 1

print(f"NextSong events written to event_datafile_new.csv: {row_count}")

pd.read_csv('event_datafile_new.csv').head()

---
## Part II — Data Modeling with Apache Cassandra

### Schema Design Philosophy

Apache Cassandra enforces **query-first design**: every table is shaped around a single query, and the `PRIMARY KEY` is derived directly from the `WHERE` and `ORDER BY` clauses of that query.

Three analytical questions drive the schema:

| # | Question | Table | Partition Key | Clustering Key |
|---|----------|-------|---------------|----------------|
| 1 | Artist, song & duration for a given session + position | `song_plays_by_session` | `sessionId` | `itemInSession` |
| 2 | Artist, song & user for a given user + session (sorted by position) | `song_plays_by_user_session` | `(userId, sessionId)` | `itemInSession` |
| 3 | All users who listened to a specific song | `users_by_song` | `song` | `userId` |

In [ ]:
cluster = Cluster(['127.0.0.1'])
session = cluster.connect()
print("Connected to Apache Cassandra cluster.")

In [ ]:
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS sparkifydb
    WITH REPLICATION = {
        'class': 'SimpleStrategy',
        'replication_factor': 1
    }
""")

session.set_keyspace('sparkifydb')
print("Keyspace 'sparkifydb' ready.")

---
### Query 1 — Song Played at a Specific Position Within a Session

**Question:** Give me the artist name, song title, and song length heard during `sessionId = 338` and `itemInSession = 4`.

```sql
SELECT artist, song, length
FROM song_plays_by_session
WHERE sessionId = 338 AND itemInSession = 4
```

**Table:** `song_plays_by_session`

**PRIMARY KEY design — `(sessionId, itemInSession)`:**
- **Partition key `sessionId`:** groups all events from the same listening session on a single Cassandra node, enabling a fast single-partition read when filtering by session.
- **Clustering column `itemInSession`:** uniquely identifies each event within a session and satisfies the `AND itemInSession = ?` predicate without needing `ALLOW FILTERING`.
- Together they form a globally unique key because no two events in the same session share the same position.

In [ ]:
session.execute("DROP TABLE IF EXISTS song_plays_by_session")

session.execute("""
    CREATE TABLE IF NOT EXISTS song_plays_by_session (
        sessionId     INT,
        itemInSession INT,
        artist        TEXT,
        song          TEXT,
        length        DECIMAL,
        PRIMARY KEY (sessionId, itemInSession)
    )
""")
print("Table 'song_plays_by_session' ready.")

In [ ]:
insert_q1 = """
    INSERT INTO song_plays_by_session (sessionId, itemInSession, artist, song, length)
    VALUES (%s, %s, %s, %s, %s)
"""

with open('event_datafile_new.csv', 'r', encoding='utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader)
    for line in csvreader:
        # event_datafile_new.csv indices:
        # 0=artist  3=itemInSession  5=length  8=sessionId  9=song
        session.execute(insert_q1, (int(line[8]), int(line[3]), line[0], line[9], float(line[5])))

print("Data inserted into 'song_plays_by_session'.")

In [ ]:
rows = session.execute("""
    SELECT artist, song, length
    FROM song_plays_by_session
    WHERE sessionId = 338 AND itemInSession = 4
""")

df_q1 = pd.DataFrame(rows, columns=['Artist', 'Song', 'Length (s)'])
print("Query 1 result — sessionId=338, itemInSession=4\n")
df_q1

---
### Query 2 — Songs Listened by a User in a Session (Ordered by Position)

**Question:** Give me the artist name, song title (sorted by `itemInSession`) and the user's full name for `userId = 10` and `sessionId = 182`.

```sql
SELECT artist, song, firstName, lastName
FROM song_plays_by_user_session
WHERE userId = 10 AND sessionId = 182
```

**Table:** `song_plays_by_user_session`

**PRIMARY KEY design — `((userId, sessionId), itemInSession)`:**
- **Composite partition key `(userId, sessionId)`:** because the query filters on *both* columns simultaneously, co-locating them in the partition key ensures a single-partition lookup with no cross-node scatter.
- **Clustering column `itemInSession`:** uniquely identifies each song within the user-session pair and causes Cassandra to store rows in ascending playback order, so the result set is naturally sorted without a separate `ORDER BY`.
- This combination uniquely identifies every song play record in the entire dataset.

In [ ]:
session.execute("DROP TABLE IF EXISTS song_plays_by_user_session")

session.execute("""
    CREATE TABLE IF NOT EXISTS song_plays_by_user_session (
        userId        INT,
        sessionId     INT,
        itemInSession INT,
        artist        TEXT,
        song          TEXT,
        firstName     TEXT,
        lastName      TEXT,
        PRIMARY KEY ((userId, sessionId), itemInSession)
    )
""")
print("Table 'song_plays_by_user_session' ready.")

In [ ]:
insert_q2 = """
    INSERT INTO song_plays_by_user_session
        (userId, sessionId, itemInSession, artist, song, firstName, lastName)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

with open('event_datafile_new.csv', 'r', encoding='utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader)
    for line in csvreader:
        # event_datafile_new.csv indices:
        # 0=artist  1=firstName  3=itemInSession  4=lastName  8=sessionId  9=song  10=userId
        session.execute(
            insert_q2,
            (int(line[10]), int(line[8]), int(line[3]), line[0], line[9], line[1], line[4])
        )

print("Data inserted into 'song_plays_by_user_session'.")

In [ ]:
rows = session.execute("""
    SELECT artist, song, firstName, lastName
    FROM song_plays_by_user_session
    WHERE userId = 10 AND sessionId = 182
""")

df_q2 = pd.DataFrame(rows, columns=['Artist', 'Song', 'First Name', 'Last Name'])
print("Query 2 result — userId=10, sessionId=182 (ordered by itemInSession)\n")
df_q2

---
### Query 3 — All Users Who Listened to a Specific Song

**Question:** Give me every user's first and last name who listened to the song `'All Hands Against His Own'`.

```sql
SELECT firstName, lastName
FROM users_by_song
WHERE song = 'All Hands Against His Own'
```

**Table:** `users_by_song`

**PRIMARY KEY design — `(song, userId)`:**
- **Partition key `song`:** all listener records for the same song are co-located on one node, making the lookup a single-partition scan and avoiding a full-table scan or `ALLOW FILTERING`.
- **Clustering column `userId`:** guarantees row uniqueness — if the same user listened to the song in multiple sessions, only one record is stored per user (the last inserted value wins), preventing duplicates in the result set.

In [ ]:
session.execute("DROP TABLE IF EXISTS users_by_song")

session.execute("""
    CREATE TABLE IF NOT EXISTS users_by_song (
        song      TEXT,
        userId    INT,
        firstName TEXT,
        lastName  TEXT,
        PRIMARY KEY (song, userId)
    )
""")
print("Table 'users_by_song' ready.")

In [ ]:
insert_q3 = """
    INSERT INTO users_by_song (song, userId, firstName, lastName)
    VALUES (%s, %s, %s, %s)
"""

with open('event_datafile_new.csv', 'r', encoding='utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader)
    for line in csvreader:
        # event_datafile_new.csv indices:
        # 1=firstName  4=lastName  9=song  10=userId
        session.execute(insert_q3, (line[9], int(line[10]), line[1], line[4]))

print("Data inserted into 'users_by_song'.")

In [ ]:
rows = session.execute("""
    SELECT firstName, lastName
    FROM users_by_song
    WHERE song = 'All Hands Against His Own'
""")

df_q3 = pd.DataFrame(rows, columns=['First Name', 'Last Name'])
print("Query 3 result — users who listened to 'All Hands Against His Own'\n")
df_q3

---
### Cleanup — Drop Tables and Close Connection

In [ ]:
session.execute("DROP TABLE IF EXISTS song_plays_by_session")
session.execute("DROP TABLE IF EXISTS song_plays_by_user_session")
session.execute("DROP TABLE IF EXISTS users_by_song")
print("All tables dropped.")

In [ ]:
session.shutdown()
cluster.shutdown()
print("Cassandra session and cluster connection closed.")